<a href="https://colab.research.google.com/github/zfs10612/researchmind/blob/main/week1/rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf sentence-transformers chromadb anthropic -q

In [ ]:
from pypdf import PdfReader

def load_and_chunk(pdf_path, chunk_size=512):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text()

    # Split into chunks by character count
    chunks = []
    for i in range(0, len(full_text), chunk_size):
        chunks.append(full_text[i:i+chunk_size])
    return chunks

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_chunks(chunks):
    return model.encode(chunks, show_progress_bar=True)

In [17]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("researchmind")

def store_chunks(chunks, embeddings):
    collection.add(
        documents=chunks,
        embeddings=embeddings.tolist(),
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )

In [18]:
import anthropic

def answer_question(question, n_results=3):
    # Embed the question
    q_embedding = model.encode([question])[0]

    # Retrieve top chunks
    results = collection.query(
        query_embeddings=[q_embedding.tolist()],
        n_results=n_results
    )
    context = "\n\n".join(results['documents'][0])

    # Ask Claude
    client_a = anthropic.Anthropic(api_key="YOUR API KEY")
    response = client_a.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }]
    )
    return response.content[0].text

In [19]:
chunks = load_and_chunk("AI research/attention_paper.pdf")
embeddings = embed_chunks(chunks)
store_chunks(chunks, embeddings)

answer = answer_question("What is the main contribution of this paper?")
print(answer)